# Three-Dimensional Ensemble-Variational (3DEnVar) Tutorial
---

## Prerequisites

<u>This tutorial requires completion of the **0.qg_tutorial_start** tutorial</u>. We will use the truth, background, and observations generated from that tutorial. 

Although this tutorial can be completed independently of other DA solver tutorials, it is also strongly recommended that you first complete and understand  **1.qg_3Dvar_tutorial** and **3.qg_LETKF_tutorial**, since the 3DEnVar is a hybridized combination of variational and ensemble DA approaches and relies upon the user being familiar with these two approaches already. 

## 1. Introduction

In the previous 3D-Var tutorial, we learned how to assimilate observations by minimizing a cost function that penalizes departures from both the background state and the observations. The background error covariance matrix $\mathbf{B}$ plays a critical role in this process by determining how information from observations spreads spatially and across variables. However, the 3D-Var method typically uses a static background error covariance, meaning that $\mathbf{B}$ does not change with the flow of the day. This static covariance is often constructed from climatological statistics and lacks the ability to capture flow-dependent error structures that vary with the current atmospheric state. Ensemble data assimilation methods, such as the Ensemble Kalman Filter (EnKF), address this by estimating $\mathbf{B}$ from an ensemble of forecasts. The ensemble spread provides a flow-dependent estimate of forecast uncertainty, which can be used to construct a dynamic background error covariance.

Three-Dimensional Ensemble-Variational data assimilation (3DEnVar) combines the strengths of both approaches. It retains the stable, well-optimized variational framework of 3D-Var with benefits such as, e.g., enforcing dynamic constraints within the cost function, while also incorporating flow-dependent error information from an ensemble. This hybrid approach has become a cornerstone of operational numerical weather prediction systems.

In this tutorial, we will explore how 3DEnVar extends the 3D-Var framework by blending static and ensemble-derived background error covariances. We will derive the mathematical formulation, implement the method in JEDI using the QG model, and examine the impact of different weighting strategies on the analysis.

#### 1.1 Hybrid Background Error Covariance

In 3DEnVar, we replace the static background error covariance $\mathbf{B}$ with a hybrid covariance that combines static and ensemble-derived components:

$$
\mathbf{B}_{hybrid} = \alpha \mathbf{B}_{static} + (1 - \alpha) (\mathcal{L} \circ \mathbf{B}_{ens})
$$

where:
- $\mathbf{B}_{static}$ is the static background error covariance (as used in 3D-Var)
- $\mathbf{B}_{ens}$ is the ensemble-derived background error covariance
- $\alpha$ is the weighting coefficient
- $\mathcal{L} \circ \mathbf{B}_{ens}$ denotes the Schur (element-wise) product of the ensemble covariance with a localization matrix $\mathcal{L}$

The weighting coefficient $\alpha$ controls the relative contributions of the static and ensemble components. When $\alpha = 1$, the method reduces to standard 3D-Var. When $\alpha = 0$, the method uses only the ensemble covariances (known as a **pure ensemble-variational** approach).

The **localization $\mathcal{L}$** is necessary because ensemble covariances estimated from a finite ensemble contain sampling errors, particularly for long-range correlations. Localization dampens spurious correlations at large distances, improving the quality of the ensemble covariance estimate.

#### 1.2 Extended Control Variable Formulation

Conceptually, 3DEnVar replaces the static covariance representation used in 3D-Var with a **control variable** expressed in the subspace spanned by the ensemble perturbations $\mathbf{X_b}$. The analysis increment can be written schematically as

<center>$\mathbf{x}_a = \mathbf{x}_b + \mathbf{X}_b \mathbf{w},$</center>

where $\mathbf{w}$ is a vector of weights determined through minimization of the variational cost function. 

In this formulation, the ensemble perturbations act as a set of basis vectors that define the directions in state space along which corrections can occur. **Because these perturbations reflect the evolving ensemble forecast, the resulting analysis increments inherit the flow-dependent covariance structure represented by the ensemble.**

An important practical advantage of 3DEnVar is that it retains the variational minimization framework. This allows the method to use existing observation operators and infrastructure developed for variational assimilation systems, while still benefiting from ensemble-derived flow-dependent covariances. In many implementations, 3DEnVar is used in a hybrid configuration, in which the ensemble covariance is combined with a static covariance model to ensure robustness and represent error structures not captured by the ensemble.


#### 1.3 Ensemble Recentering

After performing a 3DEnVar analysis, we obtain a deterministic analysis state $\mathbf{x}^{analysis}$. However, for cycling data assimilation, we also need to update the ensemble members to reflect the new analysis. 

Ensemble recentering addresses this issue by shifting the ensemble mean to match the analysis while preserving the ensemble perturbations:

$$
\mathbf{x}^{(k)}_{recentered} = \mathbf{x}^{(k)} - \bar{\mathbf{x}}^{ens} + \mathbf{x}^{analysis}
$$

where:
- $\mathbf{x}^{(k)}$ is the $k$-th original ensemble member (before recentering)
- $\bar{\mathbf{x}}^{ens}$ is the original ensemble mean
- $\mathbf{x}^{analysis}$ is the deterministic 3DEnVar analysis
- $\mathbf{x}^{(k)}_{recentered}$ is the $k$-th recentered ensemble member

This operation preserves the ensemble spread and correlation structure:

$$
\mathbf{x}^{(k)}_{recentered} - \frac{1}{K} \sum_{j=1}^{K} \mathbf{x}^{(j)}_{recentered} = \mathbf{x}^{(k)} - \bar{\mathbf{x}}^{ens}
$$

The recentered ensemble perturbations are identical to the original ensemble perturbations. This ensures that the ensemble continues to represent flow-dependent uncertainty, but centered on the improved analysis state rather than the background mean. The recentered ensemble can then be used as the initial conditions for the next forecast cycle.

Note in most advanced systems, a separate EnKF analysis is first performed on the ensemble, from which recentering is performed on the analyzed ensemble rather than the background ensemble. This allows for observations to directly update/analyze the ensemble members themselves (i.e. as in the LETKF tutorial), while recentering can then further correct any mismatches between the analyzed ensemble and the control member analysis produced by the EnVar - ensuring a stable DA system.

### Export variables and verify executables/directories

In [ ]:
export JEDI_BIN=/home/nonroot/build/bin
export JEDI_EDU=/home/nonroot/shared/EDU

Verify the executables and output directories:

In [ ]:
ls -RCd --color $JEDI_BIN/qg_4dvar.x         
ls -RCd --color $JEDI_EDU/qgEnVar/output/*

### Table of 3DEnVar experiments

| Experiment Name | 3DEnVar yaml file | Description |
| --- | --- | --- |
| exp_default | 3dvar_hybrid.yaml | Default single-observation experiment ($\alpha=0.5$)
| exp_b10e90 | 3dvar_hybrid_b10e90.yaml | Single-ob experiment, weighted towards ensemble ($\alpha=0.1$)
| exp_b90e10 | 3dvar_hybrid_b90e10.yaml | Single-ob experiment, weighted towards static-B ($\alpha=0.9$)
| exp_mult_b50e50  | 3dvar_hybrid_mult_b50e50.yaml | Assimilates 100 observations, $\alpha=0.5$
| exp_mult_b10e90  | 3dvar_hybrid_mult_b10e90.yaml | 100-obs DA, weighted towards ensemble ($\alpha=0.1$)
| exp_mult_b90e10  | 3dvar_hybrid_mult_b90e10.yaml | 100-obs DA, weighted towards static-B ($\alpha=0.9$)



## Experiments 1-3: 3DEnVar Single-Observation Experiments with Varying Weights 
***

### Step 1: Verify truth, observations, and background files exist

In [ ]:
ls $JEDI_EDU/qgstart/output/truth/*D.nc
if [ $? -ne 0 ]; then
     echo "ERROR! Truth files don't exist! Please run the 0.qg_tutorial_start tutorial first to generate the truth simulation"
fi

In [ ]:
# Verify that ensemble files were produced
ls -S $JEDI_EDU/qgstart/output/bg/bkgd*P1D.nc
if [ $? -ne 0 ]; then
     echo "ERROR! Background files don't exist! Please see the 0.qg_tutorial_start tutorial to generate the background ensemble"
fi

The above command should list out 20 ensemble members (starting with `bkgd.ens.X...` for different member numbers of `X`) as well as a single `bkgd.fc...` file that will be used as the control member for the 3DEnVar analysis.

In [ ]:
# Verify that obs file exists
ls -S $JEDI_EDU/qgstart/output/obs/truth_oneob_northeast.obs3d.nc
if [ $? -ne 0 ]; then
     echo "ERROR! Obs files doesn't exist! Please see the 0.qg_tutorial_start tutorial to generate the observations"
fi

In [ ]:
cd $JEDI_EDU
diff qg3Dvar/yamls/3dvar_oneobs_default.yaml qgEnVar/yamls/3dvar_hybrid.yaml

Before generating the observations, we should also take a moment to calculate and examine the ensemble variances of the ensemble that was just created. 

### Step 2: Run 3DEnVar expeirments!

#### Experiment 1: Default (static 0.5, ensemble 0.5)

For the default experiment, we will configure the weights to be equal between static and ensemble covariances.

Take a look at `/home/nonroot/shared/EDU/qgEnVar/yamls/3dvar_hybrid.yaml`. One thing you should notice immediately - it looks very similar to the 3dvar yaml from the **1.qg_3Dvar_tutorial**! In fact, much of the yaml file is indeed the same. For example, we are still using  ```cost type: 3D-Var```, and the `variational:` section is also the same as the 3DVar tutorial runs - the same minimizer and outer/inner loop strategies. 

The main difference in `3dvar_hybrid.yaml`with the original 3dvar yaml file (`qg3Dvar/yamls/3dvar_oneobs_default.yaml`) is pasted below:
```yaml
  background error:
    covariance model: hybrid            # activate hybrid covariance model mode
    components:  
    - covariance:                       
        covariance model: QgError        # Section that defines the static-B covariance model
        horizontal_length_scale: 2.2e6
        maximum_condition_number: 1.0e6
        standard_deviation: 1.8e7
        vertical_length_scale: 15000.0
      weight:
        value: 0.5                        # weight of static B covariances in hybrid-B
    - covariance:
        covariance model: ensemble        # Section that defines the ensemble covariances
        localization:                     # including localizations for the ensemble perturbations
          horizontal_length_scale: 4.0e6
          localization method: QG
          maximum_condition_number: 1.0e6
          standard_deviation: 1.0
          vertical_length_scale: 30000.0
        members from template:
          template:
            date: 2009-12-31T00:00:00Z
            filename: qgstart/output/bg/bkgd.ens.%mem%.2009-12-30T00:00:00Z.P1D.nc 
          pattern: %mem%
          nmembers: 20
      weight:
        value: 0.5                        # weight of Ensemble covariances in hybrid-B
```

In the above, you can see that rather than defining the QGError as the single covariance model, we have activated the `hybrid` mode. And in doing so, multiple components of the covariance model can be defined. In the above, the first section defines th static-B, which uses the same `QGError` model and parameter settings as in the `3dvar_oneobs_default.yaml` experiment from the 3DVar tutorial. 

The second section defines the `ensemble` covarianes,  read in from files defined under `members from template`. The `localization` is also applied within this section, and for this experiment we have set it to the same value `4.0e6` (4000 km) as was used in the default experiment in the LETKF tutorial

Finally, each covariance component contains a `weight` parameter that defines the weight of this component in the final hybrid covariances. Although each weight can be defined independently within JEDI, you should generally ensure that the sum of the weights add up to 1 for all components (in the above, there are two weights of 0.5 each, which do sum to 1).

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qgEnVar/yamls/3dvar_hybrid.yaml 

There will be two files in `/home/nonroot/shared/EDU/qgEnVar/output/exp_default/da`. List them with the command below, do you know what each file represents?

In [ ]:
ls $JEDI_EDU/qgEnVar/output/exp_default/da

#### Experiment 2:  Static 0.9, ensemble 0.1

First, examine the file `qgEnVar/yamls/3dvar_hybrid_e90b10.yaml`. The diff command below summarizes the differences compared to `3dvar_hybrid.yaml` for the default experiment. Besides changes to the experiment directory for outputs, we have modified the `value:` parameter under the `weight:` section in the yaml that defines the weights for each component (static and ensemble) of the hybrid covariances.

In [ ]:
cd $JEDI_EDU
diff ./qgEnVar/yamls/3dvar_hybrid.yaml ./qgEnVar/yamls/3dvar_hybrid_b90e10.yaml

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qgEnVar/yamls/3dvar_hybrid_b90e10.yaml 

Check that you have analysis and increment files produced after the DA:

In [ ]:
ls $JEDI_EDU/qgEnVar/output/exp_b90e10/da

#### Experiment 3:  Static 0.1, Ensemble 0.9

Examine the file `qgEnVar/yamls/3dvar_hybrid_b10e90.yaml`. The diff command below summarizes the differences compared to 3dvar_hybrid.yaml for the default experiment. The changes are similar to experiment 2, except we are now assigning a weight of 0.9 to the ensemble covariances, and 0.1 to the static-B covariances.

In [ ]:
cd $JEDI_EDU
diff ./qgEnVar/yamls/3dvar_hybrid.yaml ./qgEnVar/yamls/3dvar_hybrid_b10e90.yaml

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qgEnVar/yamls/3dvar_hybrid_b10e90.yaml 

Check that you have analysis and increment files produced after the DA:

In [ ]:
ls $JEDI_EDU/qgEnVar/output/exp_b10e90/da

#### Plot the results:
Finally, let's plot and compare the resulting analysis increments of each of these experiments. Note we are using the option `--plotstream` which overlays contours of streamfunction from the background field, to help assess flow-dependence.

In [ ]:
cd $JEDI_EDU/plots_scripts/
python ./plot.py qg fields \
        $JEDI_EDU/qgEnVar/output/exp_default/da/EnVar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_oneob_northeast.obs3d.nc \
        --plotstream --output $JEDI_EDU/qgEnVar/output/exp_default/plots/EnVar_inc_b50eb50 \
        --title "analysis increment, B 50% ens 50%"
python ./plot.py qg fields \
        $JEDI_EDU/qgEnVar/output/exp_b10e90/da/EnVar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_oneob_northeast.obs3d.nc \
        --plotstream --output $JEDI_EDU/qgEnVar/output/exp_b10e90/plots/EnVar_inc_b10e90 \
        --title "analysis increment, B 10% ens 90%"
python ./plot.py qg fields \
        $JEDI_EDU/qgEnVar/output/exp_b90e10/da/EnVar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_oneob_northeast.obs3d.nc \
        --plotstream --output $JEDI_EDU/qgEnVar/output/exp_b90e10/plots/EnVar_inc_b90e10 \
        --title "analysis increment, B 90% ens 10%"

Examine the produced plots (copied below as well). Can you see the effects underlying hybrid covariances have on the resulting increments? How do the increments change as the proportion of hybrid-B increases with the ensemble covariances (and decreases the static B)? 

![singleob_inc_3denvar](images/3denvar_inc_x_singleob.png)

## Experiments 4-6: Assimilate with Multiple Observations, varying the weights of static and ensemble covariances

---

Verify you have the `truth_100obs.obs3d.nc` file from the `qgstart` tutorial:

In [ ]:
cd $JEDI_EDU
ls ./qgstart/output/obs/*nc

Comparing the multiple obs experiment to the default single-obs experiment, you will see all we have changed is the observation file (obsfile) within the `obsdatain` section:

In [ ]:
cd $JEDI_EDU
diff ./qgEnVar/yamls/3dvar_hybrid.yaml ./qgEnVar/yamls/3dvar_hybrid_mult_b50e50.yaml

Let's run this experiment now:

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qgEnVar/yamls/3dvar_hybrid_mult_b50e50.yaml 

Verify that the above run executed correctly, listing the contents of the `da` directory. There should be two files produced:

In [ ]:
ls $JEDI_EDU/qgEnVar/output/exp_mult_b50e50/da

As in the single-observation experiment, we will run two additional configurations varying the weights of static B and ensemble covariance terms in the hybrid covariances. We will test one run with mostly ensemble covariances (static 10%, ensemble 90%), and another run with mostly static covariances from the QGError model (static 90%, ensemble 10%). 

You can verify these settings for each experiment looking at the below `diff` commands:

In [ ]:
cd $JEDI_EDU
diff ./qgEnVar/yamls/3dvar_hybrid_mult_b50e50.yaml ./qgEnVar/yamls/3dvar_hybrid_mult_b10e90.yaml

In [ ]:
cd $JEDI_EDU
diff ./qgEnVar/yamls/3dvar_hybrid_mult_b50e50.yaml ./qgEnVar/yamls/3dvar_hybrid_mult_b90e10.yaml

Now let's run each 3dEnVar experiment, verifying at the end both produced results:

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qgEnVar/yamls/3dvar_hybrid_mult_b10e90.yaml 

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_4dvar.x ./qgEnVar/yamls/3dvar_hybrid_mult_b90e10.yaml 

In [ ]:
echo "exp_mult_b10e90 output:"
ls -1 $JEDI_EDU/qgEnVar/output/exp_mult_b10e90/da
echo -e "\nexp_mult_b90e10 output:"
ls -1  $JEDI_EDU/qgEnVar/output/exp_mult_b90e10/da

<br>Before you look at the results, take a moment to think and form an idea of what you expect to see. After that, have a look at the results via the increment figures, generated via:

In [ ]:
cd $JEDI_EDU/plots_scripts/
python ./plot.py qg fields \
        $JEDI_EDU/qgEnVar/output/exp_mult_b50e50/da/EnVar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_100obs.obs3d.nc \
        --plotstream --output $JEDI_EDU/qgEnVar/output/exp_mult_b50e50/plots/EnVar_inc_mult_b50eb50 \
        --title "analysis increment, B 50% ens 50%"
python ./plot.py qg fields \
        $JEDI_EDU/qgEnVar/output/exp_mult_b10e90/da/EnVar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_100obs.obs3d.nc \
        --plotstream --output $JEDI_EDU/qgEnVar/output/exp_mult_b10e90/plots/EnVar_inc_mult_b10e90 \
        --title "analysis increment, B 10% ens 90%"
python ./plot.py qg fields \
        $JEDI_EDU/qgEnVar/output/exp_mult_b90e10/da/EnVar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_100obs.obs3d.nc \
        --plotstream --output $JEDI_EDU/qgEnVar/output/exp_mult_b90e10/plots/EnVar_inc_mult_b90e10 \
        --title "analysis increment, B 90% ens 10%"

![mult_inc_x](images/3denvar_mult_inc.png)

A note on the above plots: we have included the option `--plotstream`, which will plot  countour lines of streamfunction from the second file given for each plot (in this case, the background streamfunction). This can help to assess flow dependence when looking at increment figures.


In [ ]:
# Compare meananl and meanbkg errors
cd $JEDI_EDU/plots_scripts
for exp in b50e50 b10e90 b90e10; do
  python ./plot.py qg fields \
        $JEDI_EDU/qgEnVar/output/exp_mult_$exp/da/EnVar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_100obs.obs3d.nc \
        --output $JEDI_EDU/qgEnVar/output/exp_mult_$exp/plots/EnVar_anl_error \
        --title "Control analysis error" \
        >& $JEDI_EDU/qgEnVar/output/exp_mult_$exp/plots/plot_anl_err.log
  [[ $? -eq 0 ]] && echo SUCCESS PLOT ANL ERROR $exp || echo ERROR! ANL plot $exp
  python ./plot.py qg fields \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_100obs.obs3d.nc \
        --output $JEDI_EDU/qgEnVar/output/exp_mult_$exp/plots/EnVar_bkg_error \
        --title "Control background error" \
        >& $JEDI_EDU/qgEnVar/output/exp_mult_$exp/plots/plot_bkg_err.log
  [[ $? -eq 0 ]] && echo SUCCESS PLOT BKG ERROR $exp || echo ERROR! BKG plot $exp
done

![mult_err](images/3denvar_mult_err.png)

A couple questions to think about as you examine the above figures of analysis increments and errors:

- Can you see the effect of flow dependence in this case? Is it obvious, or more difficult to tell than the single-ob experiment?
- Which experiment produces the lowest analysis error on average?

The second question may be more difficult to assess visually, however we can produce RMSD statistics of model analysis/background against observations, as was performed in the LETKF tutorial with the `computermsd.sh` script. First, we need to extract analysis and background streamfunction values at the observation locations. We do this by running the `qg_hofx3d.x` program. 

Let's now run these hofx programs for each set of experiments. This can be done efficiently using a for loop for the different experiments, with the yaml files already supplied (users are encouraged to view them: `qgEnVar/yamls/hofx*yaml`) and understand the settings for each). Note the output logs of the runs are redirected to a file located at `./qgEnVar/output/exp_mult_$exp/obs`, where $\$exp$  is the unique identifier of each experiment name (e.g. b50e50).

If they run correctly, a SUCCESS message will display for each run. There should be **six** such messages (two for each experiment)

In [ ]:
cd $JEDI_EDU
for exp in b50e50 b10e90 b90e10; do
  $JEDI_BIN/qg_hofx3d.x ./qgEnVar/yamls/hofx_mult_obs_bg_$exp.yaml >& ./qgEnVar/output/exp_mult_$exp/obs/hofxbg.log
  [[ $? -eq 0 ]] && echo SUCCESS HOFX RUN BG $exp || echo ERROR! BG run $exp
  $JEDI_BIN/qg_hofx3d.x ./qgEnVar/yamls/hofx_mult_obs_an_$exp.yaml >& ./qgEnVar/output/exp_mult_$exp/obs/hofxan.log
  [[ $? -eq 0 ]] && echo SUCCESS HOFX RUN AN $exp || echo ERROR! AN run $exp
done

Next we can extract the observation info from each produced netcdf observation file into text files, using the `plot.py` scripts. We have again included a loop for brevity. Note the supplied cat commands can be uncommented to check the info in produced text files (just uncomment and change the experiment name as needed in the path)

In [ ]:
cd $JEDI_EDU/plots_scripts
for exp in b50e50 b10e90 b90e10; do
  python ./plot.py qg obs $JEDI_EDU/qgEnVar/output/exp_mult_$exp/obs/hofx_an.obs3d.nc \
        --output $JEDI_EDU/qgEnVar/output/exp_mult_$exp/plots/hofx_an \
        >& $JEDI_EDU/qgEnVar/output/exp_mult_$exp/obs/extract_hofx_an.log
  [[ $? -eq 0 ]] && echo SUCCESS EXTRACT AN INFO $exp || echo ERROR! AN $exp
  python ./plot.py qg obs $JEDI_EDU/qgEnVar/output/exp_mult_$exp/obs/hofx_bg.obs3d.nc \
        --output $JEDI_EDU/qgEnVar/output/exp_mult_$exp/plots/hofx_bg \
        >& $JEDI_EDU/qgEnVar/output/exp_mult_$exp/obs/extract_hofx_bg.log
  [[ $? -eq 0 ]] && echo SUCCESS EXTRACT BG INFO $exp || echo ERROR! BG $exp
done
#cat $JEDI_EDU/qgEnVar/output/exp_mult_b50e50/plots/hofx_an.txt
#cat $JEDI_EDU/qgEnVar/output/exp_mult_b50e50/plots/hofx_bg.txt

Now, let's compute the RMSD for each of these experiments' background and analysis files, using the extracted information in text files and the `computermsd.bash` script.

- Why do all experiments have the same RMSD of the background?
- Which experiment has the best analysis fit to observations?
- Would you use the best fitting experiment for your research? Or is this not enough information? If the latter is true, what else would you assess?

To brainstorm the last question, consider that we could run multiple DA cycles for more reliable statistics. And we could evaluate the forecast accuracy of successive first guess forecasts, to see how the analyses translate into forecasts for each. It is possible analyses with similar fits to observations may lead to substantial different forecast results, based on the underlying dynamical balance of the analyses produced by the DA.



In [ ]:
# Compute RMSD of obs for 3 experiments, compare the values
for exp in b90e10 b50e50 b10e90; do
  cd $JEDI_EDU/qgEnVar/output/exp_mult_$exp/plots
  echo -e "\n${exp}:"
  $JEDI_EDU/plots_scripts/computermsd.bash ./hofx_bg.txt ./hofx_an.txt
done
 

## Summary and Conclusions

In this tutorial, we have explored Three-Dimensional Ensemble-Variational data assimilation (3DEnVar), a hybrid method that combines the strengths of variational and ensemble data assimilation approaches. The key concepts we covered include:

1. **Motivation for Hybrid Methods:** Static background error covariances used in 3D-Var cannot capture flow-dependent error structures. Ensemble methods provide flow-dependent covariances but may lack the sophistication of variational frameworks. 3DEnVar combines both approaches.

2. **Hybrid Background Error Covariance:** The hybrid covariance is a weighted combination of static and ensemble-derived components: $\mathbf{B}_{hybrid} = \alpha \mathbf{B}_{static} + (1-\alpha) (\mathcal{L} \circ \mathbf{B}_{ens})$. The weights $\alpha$ and $(1-\alpha)$ control the relative contributions of each component.

3. **Extended Control Variable Formulation:** The analysis increment is expressed in terms of control variables that implicitly represent the hybrid covariance structure, allowing for efficient minimization without explicitly storing large covariance matrices.

4. **Localization:** Covariance localization is essential to mitigate sampling errors in the ensemble covariance. 

5. **Ensemble Recentering:** After obtaining a deterministic analysis, ensemble recentering shifts the ensemble mean to match the analysis while preserving the ensemble perturbations. This is necessary for cycling hybrid data assimilation systems.

Hybrid data assimilation methods like 3DEnVar have become operational standards in numerical weather prediction centers worldwide. They provide a practical framework for incorporating flow-dependent error information while retaining the flexibility and quality control capabilities of variational methods. As you continue your studies in data assimilation, you will encounter many extensions and refinements of the hybrid approach, including four-dimensional variants (4DEnVar) and advanced localization techniques.